In [1]:
%%writefile /kaggle/working/04_train_baseline.py
import os, time, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchvision.transforms.v2 as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

CFG = dict(
    backbone="efficientnet_b0",
    epochs=10,
    batch=4,            # studi per batch (1 studi = 3 view x 16 slice)
    lr_backbone=1e-4,
    lr_head=1e-3,
    wd=1e-2,
    warmup_frac=0.05,
    val_frac=0.10,
    seed=42,
    workers=4,
    debug=True,         # True = uji cepat (subset kecil, 1 epoch). Ubah ke False untuk training penuh
)
COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
OUT = "/kaggle/working"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]


# ---------------- util ----------------
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def find_file(name):
    """Cari file di /kaggle/input tanpa masuk ke folder kompetisi (isinya ratusan ribu DICOM)."""
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d != "competitions"]
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(f"{name} tidak ditemukan di /kaggle/input")


def mean_auc(y, p):
    """AUC per kelas; label 0.5 (uncertain) dan kosong diabaikan."""
    aucs = []
    for k in range(len(LABELS)):
        ok = ~np.isnan(y[:, k]) & (y[:, k] != 0.5)
        yk, pk = y[ok, k], p[ok, k]
        aucs.append(roc_auc_score(yk, pk) if len(np.unique(yk)) == 2 else np.nan)
    return float(np.nanmean(aucs)), aucs


# ---------------- data ----------------
class KneeDS(Dataset):
    def __init__(self, uids, ymat, vol_dir, train):
        self.uids, self.y, self.vol_dir, self.train = uids, ymat, vol_dir, train
        # Tanpa flip kiri-kanan: flip bisa menukar sisi medial/lateral
        self.affine = T.RandomAffine(degrees=10, translate=(0.06, 0.06), scale=(0.9, 1.1))

    def __len__(self):
        return len(self.uids)

    def __getitem__(self, i):
        d = np.load(os.path.join(self.vol_dir, f"{self.uids[i]}.npz"))
        vol = torch.from_numpy(d["vol"]).float() / 255.0            # (V, D, H, W)
        vmask = torch.from_numpy(d["mask"]).float()                 # (V,)
        if self.train:
            views = []
            for v in range(vol.shape[0]):
                x = self.affine(vol[v])                             # transformasi sama untuk semua slice
                x = x * random.uniform(0.85, 1.15) + random.uniform(-0.08, 0.08)
                views.append(x.clamp(0, 1))
            vol = torch.stack(views)
        y = torch.tensor(self.y[i], dtype=torch.float32)
        ym = (~torch.isnan(y)).float()
        return vol, vmask, torch.nan_to_num(y, 0.0), ym


# ---------------- model ----------------
class KneeNet(nn.Module):
    def __init__(self, backbone, n_views=3, n_out=len(LABELS), pretrained=True):
        super().__init__()
        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, in_chans=3)
        f = self.enc.num_features
        self.att = nn.ModuleList([nn.Sequential(nn.Linear(f, 128), nn.Tanh(), nn.Linear(128, 1))
                                  for _ in range(n_views)])
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(f * n_views, n_out))

    def forward(self, vol, vmask, return_att=False):
        B, V, D, H, W = vol.shape
        prev = torch.cat([vol[:, :, :1], vol[:, :, :-1]], 2)
        nxt = torch.cat([vol[:, :, 1:], vol[:, :, -1:]], 2)
        x = torch.stack([prev, vol, nxt], 3)                        # (B, V, D, 3, H, W)  2.5D
        x = (x - 0.45) / 0.225
        feats = self.enc(x.flatten(0, 2)).view(B, V, D, -1)         # (B, V, D, F)
        pooled, atts = [], []
        for v in range(V):
            a = self.att[v](feats[:, v]).softmax(1)                 # (B, D, 1) bobot per slice
            pooled.append((a * feats[:, v]).sum(1))
            atts.append(a.squeeze(-1))
        z = torch.stack(pooled, 1) * vmask[:, :, None]              # view yang hilang dinolkan
        logits = self.head(z.flatten(1))
        return (logits, torch.stack(atts, 1)) if return_att else logits


def masked_bce(logits, y, ym):
    loss = F.binary_cross_entropy_with_logits(logits, y, reduction="none")
    return (loss * ym).sum() / ym.sum().clamp(min=1)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    ps = []
    for vol, vmask, _, _ in loader:
        with torch.autocast("cuda", dtype=torch.float16):
            ps.append(torch.sigmoid(model(vol.cuda(non_blocking=True),
                                          vmask.cuda(non_blocking=True)).float()).cpu())
    return torch.cat(ps).numpy()


# ---------------- main ----------------
def main():
    seed_all(CFG["seed"])
    vol_dir = os.path.dirname(find_file("_meta.csv"))
    lab_path = find_file("report_labels.csv")
    print("Volume :", vol_dir)
    print("Label  :", lab_path)

    train_csv = pd.read_csv(os.path.join(COMP, "train.csv"))
    gold = train_csv[train_csv[LABELS].notna().all(axis=1)][["StudyInstanceUID"] + LABELS]
    gold_ids = set(gold["StudyInstanceUID"])
    have = {f[:-4] for f in os.listdir(vol_dir) if f.endswith(".npz")}

    rep = pd.read_csv(lab_path)
    rep = rep[rep["StudyInstanceUID"].isin(have) & ~rep["StudyInstanceUID"].isin(gold_ids)]
    gold = gold[gold["StudyInstanceUID"].isin(have)].reset_index(drop=True)
    tr, va = train_test_split(rep, test_size=CFG["val_frac"], random_state=CFG["seed"])
    if CFG["debug"]:
        tr, va, CFG["epochs"] = tr.head(64), va.head(32), 1
    print(f"Train {len(tr)} | Val {len(va)} | Gold holdout {len(gold)}")

    def loader(df, train):
        ds = KneeDS(df["StudyInstanceUID"].tolist(), df[LABELS].values.astype(np.float32),
                    vol_dir, train)
        return DataLoader(ds, batch_size=CFG["batch"], shuffle=train, drop_last=train,
                          num_workers=CFG["workers"], pin_memory=True,
                          persistent_workers=CFG["workers"] > 0)

    dl_tr, dl_va, dl_gold = loader(tr, True), loader(va, False), loader(gold, False)

    net = KneeNet(CFG["backbone"]).cuda().to(memory_format=torch.channels_last)
    model = nn.DataParallel(net) if torch.cuda.device_count() > 1 else net
    opt = torch.optim.AdamW([
        {"params": net.enc.parameters(), "lr": CFG["lr_backbone"]},
        {"params": list(net.att.parameters()) + list(net.head.parameters()), "lr": CFG["lr_head"]},
    ], weight_decay=CFG["wd"])
    total = CFG["epochs"] * len(dl_tr)
    warm = max(1, int(total * CFG["warmup_frac"]))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda")

    best, log = -1.0, []
    for ep in range(CFG["epochs"]):
        model.train()
        t0, run = time.time(), 0.0
        for it, (vol, vmask, y, ym) in enumerate(dl_tr):
            vol, vmask, y, ym = (t.cuda(non_blocking=True) for t in (vol, vmask, y, ym))
            with torch.autocast("cuda", dtype=torch.float16):
                loss = masked_bce(model(vol, vmask).float(), y, ym)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item()
            if it % 100 == 0:
                print(f"  ep{ep} it{it}/{len(dl_tr)} loss {run/(it+1):.4f} "
                      f"({(time.time()-t0)/(it+1):.2f} s/it)", flush=True)

        va_auc, _ = mean_auc(va[LABELS].values.astype(np.float32), predict(model, dl_va))
        gold_auc, _ = mean_auc(gold[LABELS].values.astype(np.float32), predict(model, dl_gold))
        log.append({"epoch": ep, "loss": run / len(dl_tr), "val_auc": va_auc,
                    "gold_auc": gold_auc, "min": (time.time() - t0) / 60})
        print(f"Epoch {ep}: loss {run/len(dl_tr):.4f} | val AUC {va_auc:.4f} | "
              f"gold AUC {gold_auc:.4f} | {(time.time()-t0)/60:.1f} menit", flush=True)
        if va_auc > best:                                   # checkpoint dipilih dari val, bukan gold
            best = va_auc
            torch.save(net.state_dict(), os.path.join(OUT, f"{CFG['backbone']}_best.pt"))
        pd.DataFrame(log).to_csv(os.path.join(OUT, "train_log.csv"), index=False)

    # -------- evaluasi akhir checkpoint terbaik pada 58 label emas --------
    net.load_state_dict(torch.load(os.path.join(OUT, f"{CFG['backbone']}_best.pt")))
    p_gold = predict(model, dl_gold)
    g_auc, per = mean_auc(gold[LABELS].values.astype(np.float32), p_gold)
    print(f"\n=== Checkpoint terbaik (val AUC {best:.4f}) pada label emas ===")
    for l, a in zip(LABELS, per):
        print(f"{l:18s} AUC={a:.3f}")
    print(f"Mean AUC gold: {g_auc:.4f}")
    pd.concat([gold[["StudyInstanceUID"]], pd.DataFrame(p_gold, columns=LABELS)], axis=1) \
      .to_csv(os.path.join(OUT, "gold_preds.csv"), index=False)
    os._exit(0)   # hindari proses DataLoader yang menggantung saat keluar


if __name__ == "__main__":
    main()

Writing /kaggle/working/04_train_baseline.py


In [2]:
!python /kaggle/working/04_train_baseline.py

Volume : /kaggle/input/notebooks/kellenvalerie/rsna-knee-02-preprocess-mri/vol_train
Label  : /kaggle/input/datasets/kellenvalerie/rsna-knee-report-labels/report_labels.csv
Train 64 | Val 32 | Gold holdout 58

model.safetensors: downloading bytes:                       |  0.00B            
model.safetensors: downloading bytes: ███▌                  | 3.50MB            
model.safetensors: downloading bytes: █████████▌            | 9.31MB,  345kB/s  
model.safetensors: downloading bytes: ██████████████▍       | 14.0MB,  912kB/s  
model.safetensors: downloading bytes: ██████████████████████| 20.2MB, 1.94MB/s  
model.safetensors: reconstructing file: 100%|██████| 21.4MB / 21.4MB, 2.06MB/s  
  ep0 it0/16 loss 0.7032 (19.43 s/it)
Epoch 0: loss 0.5736 | val AUC 0.5435 | gold AUC 0.5240 | 5.7 menit

=== Checkpoint terbaik (val AUC 0.5435) pada label emas ===
ACL                AUC=0.382
MCL                AUC=0.510
Medial Meniscus    AUC=0.546
Lateral Meniscus   AUC=0.551
Medial OA          AU

In [3]:
%%writefile /kaggle/working/04_train_baseline.py
import os, time, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchvision.transforms.v2 as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

CFG = dict(
    backbone="efficientnet_b0",
    epochs=10,
    batch=4,            # studi per batch (1 studi = 3 view x 16 slice)
    lr_backbone=1e-4,
    lr_head=1e-3,
    wd=1e-2,
    warmup_frac=0.05,
    val_frac=0.10,
    seed=42,
    workers=4,
    debug=False,        # True = uji cepat (subset kecil, 1 epoch)
)
COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
OUT = "/kaggle/working"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]


# ---------------- util ----------------
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def find_file(name):
    """Cari file di /kaggle/input tanpa masuk ke folder kompetisi (isinya ratusan ribu DICOM)."""
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d != "competitions"]
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(f"{name} tidak ditemukan di /kaggle/input")


def mean_auc(y, p):
    """AUC per kelas; label 0.5 (uncertain) dan kosong diabaikan."""
    aucs = []
    for k in range(len(LABELS)):
        ok = ~np.isnan(y[:, k]) & (y[:, k] != 0.5)
        yk, pk = y[ok, k], p[ok, k]
        aucs.append(roc_auc_score(yk, pk) if len(np.unique(yk)) == 2 else np.nan)
    return float(np.nanmean(aucs)), aucs


# ---------------- data ----------------
class KneeDS(Dataset):
    def __init__(self, uids, ymat, vol_dir, train):
        self.uids, self.y, self.vol_dir, self.train = uids, ymat, vol_dir, train
        # Tanpa flip kiri-kanan: flip bisa menukar sisi medial/lateral
        self.affine = T.RandomAffine(degrees=10, translate=(0.06, 0.06), scale=(0.9, 1.1))

    def __len__(self):
        return len(self.uids)

    def __getitem__(self, i):
        d = np.load(os.path.join(self.vol_dir, f"{self.uids[i]}.npz"))
        vol = torch.from_numpy(d["vol"]).float() / 255.0            # (V, D, H, W)
        vmask = torch.from_numpy(d["mask"]).float()                 # (V,)
        if self.train:
            views = []
            for v in range(vol.shape[0]):
                x = self.affine(vol[v])                             # transformasi sama untuk semua slice
                x = x * random.uniform(0.85, 1.15) + random.uniform(-0.08, 0.08)
                views.append(x.clamp(0, 1))
            vol = torch.stack(views)
        y = torch.tensor(self.y[i], dtype=torch.float32)
        ym = (~torch.isnan(y)).float()
        return vol, vmask, torch.nan_to_num(y, 0.0), ym


# ---------------- model ----------------
class KneeNet(nn.Module):
    def __init__(self, backbone, n_views=3, n_out=len(LABELS), pretrained=True):
        super().__init__()
        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, in_chans=3)
        f = self.enc.num_features
        self.att = nn.ModuleList([nn.Sequential(nn.Linear(f, 128), nn.Tanh(), nn.Linear(128, 1))
                                  for _ in range(n_views)])
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(f * n_views, n_out))

    def forward(self, vol, vmask, return_att=False):
        B, V, D, H, W = vol.shape
        prev = torch.cat([vol[:, :, :1], vol[:, :, :-1]], 2)
        nxt = torch.cat([vol[:, :, 1:], vol[:, :, -1:]], 2)
        x = torch.stack([prev, vol, nxt], 3)                        # (B, V, D, 3, H, W)  2.5D
        x = (x - 0.45) / 0.225
        feats = self.enc(x.flatten(0, 2)).view(B, V, D, -1)         # (B, V, D, F)
        pooled, atts = [], []
        for v in range(V):
            a = self.att[v](feats[:, v]).softmax(1)                 # (B, D, 1) bobot per slice
            pooled.append((a * feats[:, v]).sum(1))
            atts.append(a.squeeze(-1))
        z = torch.stack(pooled, 1) * vmask[:, :, None]              # view yang hilang dinolkan
        logits = self.head(z.flatten(1))
        return (logits, torch.stack(atts, 1)) if return_att else logits


def masked_bce(logits, y, ym):
    loss = F.binary_cross_entropy_with_logits(logits, y, reduction="none")
    return (loss * ym).sum() / ym.sum().clamp(min=1)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    ps = []
    for vol, vmask, _, _ in loader:
        with torch.autocast("cuda", dtype=torch.float16):
            ps.append(torch.sigmoid(model(vol.cuda(non_blocking=True),
                                          vmask.cuda(non_blocking=True)).float()).cpu())
    return torch.cat(ps).numpy()


# ---------------- main ----------------
def main():
    seed_all(CFG["seed"])
    vol_dir = os.path.dirname(find_file("_meta.csv"))
    lab_path = find_file("report_labels.csv")
    print("Volume :", vol_dir)
    print("Label  :", lab_path)

    train_csv = pd.read_csv(os.path.join(COMP, "train.csv"))
    gold = train_csv[train_csv[LABELS].notna().all(axis=1)][["StudyInstanceUID"] + LABELS]
    gold_ids = set(gold["StudyInstanceUID"])
    have = {f[:-4] for f in os.listdir(vol_dir) if f.endswith(".npz")}

    rep = pd.read_csv(lab_path)
    rep = rep[rep["StudyInstanceUID"].isin(have) & ~rep["StudyInstanceUID"].isin(gold_ids)]
    gold = gold[gold["StudyInstanceUID"].isin(have)].reset_index(drop=True)
    tr, va = train_test_split(rep, test_size=CFG["val_frac"], random_state=CFG["seed"])
    if CFG["debug"]:
        tr, va, CFG["epochs"] = tr.head(64), va.head(32), 1
    print(f"Train {len(tr)} | Val {len(va)} | Gold holdout {len(gold)}")

    def loader(df, train):
        ds = KneeDS(df["StudyInstanceUID"].tolist(), df[LABELS].values.astype(np.float32),
                    vol_dir, train)
        return DataLoader(ds, batch_size=CFG["batch"], shuffle=train, drop_last=train,
                          num_workers=CFG["workers"], pin_memory=True,
                          persistent_workers=CFG["workers"] > 0)

    dl_tr, dl_va, dl_gold = loader(tr, True), loader(va, False), loader(gold, False)

    # 1 GPU saja: DataParallel di Kaggle T4 x2 (tanpa P2P) ~17x lebih lambat
    torch.backends.cudnn.benchmark = True
    net = KneeNet(CFG["backbone"]).cuda().to(memory_format=torch.channels_last)
    model = net
    opt = torch.optim.AdamW([
        {"params": net.enc.parameters(), "lr": CFG["lr_backbone"]},
        {"params": list(net.att.parameters()) + list(net.head.parameters()), "lr": CFG["lr_head"]},
    ], weight_decay=CFG["wd"])
    total = CFG["epochs"] * len(dl_tr)
    warm = max(1, int(total * CFG["warmup_frac"]))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda")

    best, log = -1.0, []
    for ep in range(CFG["epochs"]):
        model.train()
        t0, run = time.time(), 0.0
        for it, (vol, vmask, y, ym) in enumerate(dl_tr):
            vol, vmask, y, ym = (t.cuda(non_blocking=True) for t in (vol, vmask, y, ym))
            with torch.autocast("cuda", dtype=torch.float16):
                loss = masked_bce(model(vol, vmask).float(), y, ym)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item()
            if it % 100 == 0:
                print(f"  ep{ep} it{it}/{len(dl_tr)} loss {run/(it+1):.4f} "
                      f"({(time.time()-t0)/(it+1):.2f} s/it)", flush=True)

        va_auc, _ = mean_auc(va[LABELS].values.astype(np.float32), predict(model, dl_va))
        gold_auc, _ = mean_auc(gold[LABELS].values.astype(np.float32), predict(model, dl_gold))
        log.append({"epoch": ep, "loss": run / len(dl_tr), "val_auc": va_auc,
                    "gold_auc": gold_auc, "min": (time.time() - t0) / 60})
        print(f"Epoch {ep}: loss {run/len(dl_tr):.4f} | val AUC {va_auc:.4f} | "
              f"gold AUC {gold_auc:.4f} | {(time.time()-t0)/60:.1f} menit", flush=True)
        if va_auc > best:                                   # checkpoint dipilih dari val, bukan gold
            best = va_auc
            torch.save(net.state_dict(), os.path.join(OUT, f"{CFG['backbone']}_best.pt"))
        pd.DataFrame(log).to_csv(os.path.join(OUT, "train_log.csv"), index=False)

    # -------- evaluasi akhir checkpoint terbaik pada 58 label emas --------
    net.load_state_dict(torch.load(os.path.join(OUT, f"{CFG['backbone']}_best.pt")))
    p_gold = predict(model, dl_gold)
    g_auc, per = mean_auc(gold[LABELS].values.astype(np.float32), p_gold)
    print(f"\n=== Checkpoint terbaik (val AUC {best:.4f}) pada label emas ===")
    for l, a in zip(LABELS, per):
        print(f"{l:18s} AUC={a:.3f}")
    print(f"Mean AUC gold: {g_auc:.4f}")
    pd.concat([gold[["StudyInstanceUID"]], pd.DataFrame(p_gold, columns=LABELS)], axis=1) \
      .to_csv(os.path.join(OUT, "gold_preds.csv"), index=False)
    os._exit(0)   # hindari proses DataLoader yang menggantung saat keluar


if __name__ == "__main__":
    main()

Overwriting /kaggle/working/04_train_baseline.py


In [4]:
!CUDA_VISIBLE_DEVICES=0 python /kaggle/working/04_train_baseline.py

Volume : /kaggle/input/notebooks/kellenvalerie/rsna-knee-02-preprocess-mri/vol_train
Label  : /kaggle/input/datasets/kellenvalerie/rsna-knee-report-labels/report_labels.csv
Train 3914 | Val 435 | Gold holdout 58
  ep0 it0/978 loss 0.6979 (73.18 s/it)
  ep0 it100/978 loss 0.5664 (1.12 s/it)
  ep0 it200/978 loss 0.5320 (0.78 s/it)
  ep0 it300/978 loss 0.5171 (0.66 s/it)
  ep0 it400/978 loss 0.5081 (0.61 s/it)
  ep0 it500/978 loss 0.5018 (0.58 s/it)
  ep0 it600/978 loss 0.4965 (0.56 s/it)
  ep0 it700/978 loss 0.4927 (0.55 s/it)
  ep0 it800/978 loss 0.4862 (0.54 s/it)
  ep0 it900/978 loss 0.4827 (0.53 s/it)
Epoch 0: loss 0.4794 | val AUC 0.7398 | gold AUC 0.7652 | 9.4 menit
  ep1 it0/978 loss 0.4551 (0.96 s/it)
  ep1 it100/978 loss 0.4365 (0.47 s/it)
  ep1 it200/978 loss 0.4370 (0.47 s/it)
  ep1 it300/978 loss 0.4340 (0.47 s/it)
  ep1 it400/978 loss 0.4324 (0.47 s/it)
  ep1 it500/978 loss 0.4258 (0.47 s/it)
  ep1 it600/978 loss 0.4227 (0.47 s/it)
  ep1 it700/978 loss 0.4205 (0.47 s/it)
  e